# Forecasting Pipeline – SARIMA Model

This notebook documents the forecasting pipeline used to predict future CPU and RAM utilization for the data center optimization project.

The forecasting stage receives the processed dataset from the data cleaning and feature engineering branch and generates forecast outputs that will be used by the optimization model.

## Forecasting Scope

Input:

`data/processed/forecasting_dataset.parquet`

Main tasks:

1. Prepare forecasting data.
2. Aggregate time-based forecasting windows.
3. Train SARIMA forecasting models.
4. Evaluate forecast accuracy.
5. Generate CPU and RAM workload predictions.
6. Prepare optimization-ready forecast outputs.

Expected outputs:

`data/processed/forecast_predictions.parquet`

`data/processed/optimization_input_dataset.parquet`

In [1]:
# Standard libraries
from pathlib import Path
import warnings

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt

# Forecasting
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Metrics
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
)

warnings.filterwarnings("ignore")

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [ ]:
# Project paths

DATA_PATH = Path("data/interim/cleaned_data.parquet")

SARIMA_READY_PATH = Path(
    "data/processed/sarima_ready_dataset.parquet"
)

FORECAST_OUTPUT_PATH = Path(
    "data/processed/forecast_predictions.parquet"
)

OPTIMIZATION_OUTPUT_PATH = Path(
    "data/processed/optimization_input_dataset.parquet"
)

MODEL_PATH = Path("models/forecasting")

print("Project paths configured successfully.")

Project paths configured successfully.


In [3]:
# Check if the processed forecasting dataset is available

if DATA_PATH.exists():
    print(f"Dataset found: {DATA_PATH}")
else:
    print(f"Dataset not found yet: {DATA_PATH}")
    print("This is expected if the data cleaning and feature engineering branch is not completed yet.")

Dataset not found yet: data\processed\forecasting_dataset.parquet
This is expected if the data cleaning and feature engineering branch is not completed yet.


## SARIMA Forecasting Approach

The forecasting pipeline uses Seasonal AutoRegressive Integrated Moving Average (SARIMA) models to predict future CPU and RAM utilization patterns.

SARIMA was selected because:

- The dataset is expected to contain time-series workload behavior.
- CPU and RAM utilization typically present temporal dependencies and seasonality.
- SARIMA performs well for interpretable short-term forecasting tasks.
- The model is computationally efficient and suitable for optimization-driven pipelines.

The forecasting workflow includes:

1. Preparing time-series forecasting windows.
2. Aggregating utilization metrics by hourly intervals.
3. Training SARIMA models for CPU and RAM utilization.
4. Evaluating forecast accuracy using MAE, RMSE, and MAPE.
5. Generating future workload predictions.
6. Producing optimization-ready forecast outputs.